# Crear les dades inicials amb els paràmetres inicials

In [53]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import numpy as np

In [54]:
def simulate_tour(tour_df:pd.DataFrame, players_df:pd.DataFrame, k, ksi, s, initial_elo, min_games, year_to_simulate, rmv_retired): 
    assert tour_df.isna().sum().sum() == 0, f'nan values in tour_df\n{tour_df.isna().sum()}'

    total_matches = len(tour_df)

    tour_df = tour_df.sort_values(by='tourney_date', ascending=True)

    if year_to_simulate != 'Tots': 
        tour_df = tour_df[tour_df['tour_year']==year_to_simulate]

    all_players_dic = {player_id: {
            'player_id': player_id, 
            'elo_rating': initial_elo, 
            'elo_clay_rating': initial_elo, 
            'elo_hard_rating': initial_elo, 
            'elo_grass_rating': initial_elo, 
            'elo_carpet_rating': initial_elo, 
            'elo_unknown_rating': initial_elo,
            'max_elo_rating': initial_elo,
            'min_elo_rating': initial_elo,
            'n_games': 0,
            'last_game': None,
            'n_wins': 0,
            'n_losses': 0,
            'n_titles': 0
        } for player_id in players_df['player_id']}

    elo_history_list = []

    year = ''
    for _, m in tour_df.iterrows():

        # Update elo ratings
        match s:
            case 'delta':
                Sw = 1
                Sl = 0

            case 'thirds': 
                if m['best_of'] != m['num_sets'] or m['best_of'] == 1:
                    Sw = 1
                    Sl = 0
                else: 
                    Sw = 2/3
                    Sl = 1/3

        
        # Algorisme per calcular elo-ratings
        winner_id = m['winner_id']
        loser_id = m['loser_id']
        elo_surface = f'elo_{m['surface'].lower()}_rating'
        match_date = m['tourney_date']
        surface = m['surface']

        old_wr =  all_players_dic[winner_id]['elo_rating']
        old_lr = all_players_dic[loser_id]['elo_rating']
        # Surface
        old_slr = all_players_dic[winner_id][elo_surface]
        old_swr = all_players_dic[loser_id][elo_surface]

        mu_w = 1 / (1 + pow(10, -(old_wr - old_lr)/ksi))
        mu_l = 1 / (1 + pow(10, -(old_lr - old_wr)/ksi))
        # Surface
        mu_sw = 1 / (1 + pow(10, -(old_swr - old_slr)/ksi))
        mu_sl = 1 / (1 + pow(10, -(old_slr - old_swr)/ksi))

        # Actualitzar els valors dels elo-ratings dels jugadors. 
        winner_new_elo = old_wr + k*(Sw - mu_w)
        loser_new_elo = old_lr + k*(Sl - mu_l)
        all_players_dic[winner_id]['elo_rating'] = winner_new_elo
        all_players_dic[loser_id]['elo_rating'] = loser_new_elo
        # Surface
        all_players_dic[winner_id][elo_surface] = old_swr + k*(Sw - mu_sw)
        all_players_dic[loser_id][elo_surface] = old_slr + k*(Sl - mu_sl)

        # Stats
        all_players_dic[loser_id]['n_games'] += 1
        all_players_dic[loser_id]['last_game'] = match_date
        all_players_dic[winner_id]['n_games'] += 1
        all_players_dic[winner_id]['last_game'] = match_date
        all_players_dic[winner_id]['n_wins'] += 1
        all_players_dic[loser_id]['n_losses'] += 1
        if winner_new_elo > all_players_dic[winner_id]['max_elo_rating']:
            all_players_dic[winner_id]['max_elo_rating'] = winner_new_elo
        if loser_new_elo < all_players_dic[loser_id]['min_elo_rating']:
            all_players_dic[loser_id]['min_elo_rating'] = loser_new_elo
        if m['round'] == 'F':
            all_players_dic[winner_id]['n_titles'] += 1
        assert all_players_dic[winner_id]['n_games'] == all_players_dic[winner_id]['n_wins'] + all_players_dic[winner_id]['n_losses'],\
            f"Error: n_games != n_wins + n_losses -> {all_players_dic[winner_id]['n_games']} != {all_players_dic[winner_id]['n_wins']} + {all_players_dic[winner_id]['n_losses']}"

        # Històric
        winner_history = {
            'player_id': winner_id,
            'date': match_date,
            'elo_rating': winner_new_elo
        }
        loser_history = {
            'player_id': loser_id,
            'date': match_date,
            'elo_rating': loser_new_elo
        }

        elo_history_list.append(winner_history)
        elo_history_list.append(loser_history)


    players_simulated: pd.DataFrame = pd.DataFrame.from_dict(all_players_dic, orient='index')

    ranking: pd.DataFrame = players_df.merge(players_simulated, how='left', on='player_id')\
        .sort_values(by='elo_rating', ascending=False)\
        .reset_index(drop=True)\
        .drop(columns=['elo_unknown_rating'])\
        .round(0)

    # Filtrar mínim de partits
    ranking_filtered = ranking[(ranking['n_games']>min_games)]
    ranking_no_filtered = ranking.copy()
    
    # Filtrar retirats
    if rmv_retired:
        ranking_filtered = ranking[(ranking['last_game'] > '2024-01-01')].reset_index(drop=True)

    ranking_filtered['rank'] = ranking_filtered.index + 1

    # Diccionari {player_id: rank} de ranking_filtered
    rank_mapping = ranking_filtered.set_index('player_id')['rank'].to_dict()
    # Assignem el rank al df no filtrat
    ranking_no_filtered['rank'] = ranking_no_filtered['player_id'].map(rank_mapping)
    # Els jugadors que no estan al ranking filtrat no tenen rank ('-')
    ranking_no_filtered['rank'] = ranking_no_filtered['rank'].fillna(-1).astype(int)
    ranking_no_filtered['rank'] = ranking_no_filtered['rank'].replace(-1, '-')

    elo_history_df = pd.DataFrame(elo_history_list)\
                        .astype({'date': 'datetime64[ns]'})
    
    ## COSES DEL RANKING HISTORIC QUE NO FAIG SERVIR: ################################################################################
    # Funciona però està malament perquè en cada date-block només apareixen al ranking els jugadors que han jugat en aquella data. 

    # Crear diccionari {tour_year: [players_id]} amb els players_id unics de cada tour
    unique_years = np.sort(tour_df['tour_year'].unique())
    tour_years_dic = {}
    for year in unique_years: 
        tour_years_dic[year] = pd.concat([
                tour_df[tour_df['tour_year'] == year]['winner_id'],
                tour_df[tour_df['tour_year'] == year]['loser_id']
            ]).unique()
        


    # Agrupar elo_history_df per data i player id, quedar-se amb l'última aparició si hi ha duplicats de date-player_id
    elo_rankings_df = elo_history_df.groupby(['date', 'player_id']).last().reset_index()

    # Filtrar cada grup de dates per els player_id que estan al diccionary
    elo_rankings_df_filtered = pd.DataFrame()
    for year, players_id in tour_years_dic.items(): 

        # Agafem el block d'un mateix tour
        year_block = elo_rankings_df[elo_rankings_df['date'].dt.year == year]
        
        # Filtrem pels jugadors que han jugat aquest tour
        year_block = year_block[year_block['player_id'].isin(players_id)]

        # Concatenem el block al dataframe
        elo_rankings_df_filtered = pd.concat([elo_rankings_df_filtered, year_block]).reset_index(drop=True)

    # Per cada data calcular el rank i afegir la columna
    elo_rankings_df['rank'] = elo_rankings_df.groupby('date')['elo_rating'].rank(ascending=False, method='dense').astype(int)
    ##############################################################################################################################

    return ranking_filtered, ranking_no_filtered, elo_history_df, elo_rankings_df.sort_values(by=['date', 'rank'], ascending=[True, True]).reset_index(drop=True)


In [55]:
def simulate_tour_optimized(tour_df:pd.DataFrame, players_df:pd.DataFrame, k, xi, s, initial_elo, min_games, year_to_simulate): 

    # TODO: Triga moltissim i no calcula bé el rank. La resta (Sembla) que tot esta okay
    assert tour_df.isna().sum().sum() == 0, f'nan values in tour_df\n{tour_df.isna().sum()}'

    tour_df = tour_df.sort_values(by='tourney_date', ascending=True)

    if year_to_simulate != 'Tots': 
        tour_df = tour_df[tour_df['tour_year']==year_to_simulate]

    all_players_dic = {player_id: {
            'player_id': player_id, 
            'elo_rating': initial_elo, 
            'elo_clay_rating': initial_elo, 
            'elo_hard_rating': initial_elo, 
            'elo_grass_rating': initial_elo, 
            'elo_carpet_rating': initial_elo, 
            'elo_unknown_rating': initial_elo,
            'n_games': 0,
            'last_game': None,
            'n_wins': 0,
            'n_losses': 0
        } for player_id in players_df['player_id']}
    
    elo_rankings_df = pd.DataFrame()
    elo_history_list = []
    unique_years = np.sort(tour_df['tour_year'].unique())
    
    # Bucle principal
    for year in unique_years:
        print(year)
        
        # Jugadors unics del year_block
        unique_p_ids = pd.concat([
                tour_df[tour_df['tour_year'] == year]['winner_id'],
                tour_df[tour_df['tour_year'] == year]['loser_id']
            ]).unique()

        # Dates uniques del year_block
        unique_dates = np.sort(tour_df[tour_df['tour_year']==year]['tourney_date'].unique())
        print(unique_dates)
        for date in unique_dates:             
            # Date block per guardar resultats i concatenar al final
            date_block_dict = {player_id: {
                'player_id': player_id,
                'elo_rating': all_players_dic[player_id]['elo_rating'],
                'elo_clay_rating': all_players_dic[player_id]['elo_clay_rating'],
                'elo_hard_rating': all_players_dic[player_id]['elo_hard_rating'],
                'elo_grass_rating': all_players_dic[player_id]['elo_grass_rating'],
                'elo_carpet_rating': all_players_dic[player_id]['elo_carpet_rating'],
                'elo_unknown_rating': all_players_dic[player_id]['elo_unknown_rating'],
            } for player_id in unique_p_ids}
                
            # Date block sobre el qual iterarem
            date_block_df: pd.DataFrame = tour_df[(tour_df['tourney_date'] == date) & (tour_df['tour_year'] == year)] # Fer el match del year no cal pk la date ja es més restrictiu
            if date_block_df.empty:
                print(f"date_block_df is empty for date {date} and year {year}")
                exit(0)
            for _, m in date_block_df.iterrows(): 

                # Obtenir algunes dades
                elo_surface = f'elo_{m['surface'].lower()}_rating'
                old_wr = date_block_dict[m['winner_id']]['elo_rating']
                old_swr = date_block_dict[m['winner_id']][elo_surface]
                old_lr = date_block_dict[m['loser_id']]['elo_rating']
                old_slr = date_block_dict[m['loser_id']][elo_surface]

                winner_id = m['winner_id']
                loser_id = m['loser_id']
                elo_surface = f'elo_{m['surface'].lower()}_rating'
                match_date = m['tourney_date']
                surface = m['surface']
                
                match s:
                    case 'delta':
                        Sw = 1
                        Sl = 0

                    case 'thirds': 
                        if m['best_of'] != m['num_sets'] or m['best_of'] == 1:
                            Sw = 1
                            Sl = 0
                        else: 
                            Sw = 2/3
                            Sl = 1/3

                # Algorisme #
                mu_w = 1 / (1 + pow(10, -(old_wr - old_lr)/xi))
                mu_l = 1 / (1 + pow(10, -(old_lr - old_wr)/xi))
                # Surface
                mu_sw = 1 / (1 + pow(10, -(old_swr - old_slr)/xi))
                mu_sl = 1 / (1 + pow(10, -(old_slr - old_swr)/xi))

                # Actualitzar els valors dels elo-ratings dels jugadors. 
                winner_new_elo = old_wr + k*(Sw - mu_w)
                loser_new_elo = old_lr + k*(Sl - mu_l)
                winner_new_s_elo = old_swr + k*(Sw - mu_sw)
                loser_new_s_elo = old_slr + k*(Sl - mu_sl)

                # Guardem resultats al date_block
                date_block_dict[winner_id]['elo_rating'] = winner_new_elo
                date_block_dict[loser_id]['elo_rating'] = loser_new_elo
                date_block_dict[winner_id][elo_surface] = winner_new_s_elo
                date_block_dict[loser_id][elo_surface] = loser_new_s_elo

                # Guardem resultats al diccionari de jugadors
                all_players_dic[winner_id]['elo_rating'] = winner_new_elo
                all_players_dic[loser_id]['elo_rating'] = loser_new_elo
                all_players_dic[winner_id][elo_surface] = winner_new_s_elo
                all_players_dic[loser_id][elo_surface] = loser_new_s_elo

                # Stats #
                all_players_dic[loser_id]['n_games'] += 1
                all_players_dic[loser_id]['last_game'] = match_date
                all_players_dic[winner_id]['n_games'] += 1
                all_players_dic[winner_id]['last_game'] = match_date
                all_players_dic[winner_id]['n_wins'] += 1
                all_players_dic[loser_id]['n_losses'] += 1
                assert all_players_dic[winner_id]['n_games'] == all_players_dic[winner_id]['n_wins'] + all_players_dic[winner_id]['n_losses'],\
                    f"Error: n_games != n_wins + n_losses -> {all_players_dic[winner_id]['n_games']} != {all_players_dic[winner_id]['n_wins']} + {all_players_dic[winner_id]['n_losses']}"


                # Històric (com un jugador pot jugar més d'un partit en un mateix dia, al data_block només s'hi guardarà l'elo amb què acaba el dia)
                # Per al plot creo una llista amb tots els elo-ratings de cada jugador
                winner_history = {
                    'player_id': winner_id,
                    'date': match_date,
                    'elo_rating': winner_new_elo
                }
                loser_history = {
                    'player_id': loser_id,
                    'date': match_date,
                    'elo_rating': loser_new_elo
                }

                elo_history_list.append(winner_history)
                elo_history_list.append(loser_history)

            # Concatenem el ranking block en un dataframe
            ranking_block_df = pd.DataFrame.from_dict(date_block_dict, orient='index')\
                .sort_values(by='elo_rating')
            
            # Afegim columnes date i rank i concatenem al ranking_historic
            ranking_block_df['date'] = date
            ranking_block_df['rank'] = ranking_block_df.index + 1
            elo_rankings_df = pd.concat([elo_rankings_df, ranking_block_df], ignore_index=True)


    # Passar all_players_dic a dataframe
    all_players_df: pd.DataFrame = pd.DataFrame.from_dict(all_players_dic, orient='index')

    # Fer merge amb el df de tots els jugadors per tenir totes les stats
    ranking: pd.DataFrame = players_df.merge(all_players_df, how='left', on='player_id')\
        .sort_values(by='elo_rating', ascending=False)\
        .reset_index(drop=True)\
        .drop(columns=['elo_unknown_rating'])\
        .round(0)

    # Filtrar per minim de partits
    ranking_filtered = ranking[(ranking['n_games']>min_games)]# & (ranking['last_game'] > '2023-01-01')].reset_index(drop=True)

    # Crear columna rank
    ranking_filtered['rank'] = ranking_filtered.index + 1

    # Passar la llista de diccionaris amb l'historic a un dataframe i canviar format de la data
    elo_history_df = pd.DataFrame(elo_history_list)\
                        .astype({'date': 'datetime64[ns]'}).reset_index(drop=True)
    
    return ranking_filtered, elo_history_df, elo_rankings_df


In [56]:
def calcula_ranking_historic(tour_df, elo_history_df):
    unique_dates = tour_df['tourney_date'].unique()
    unique_years = tour_df['tour_year'].unique()
    unique_dates.sort()
    unique_years.sort()
    
    for year in unique_years:
        players_in_year = pd.concat([
            tour_df[tour_df['tour_year'] == year]['winner_id'],
            tour_df[tour_df['tour_year'] == year]['loser_id']
        ]).unique()

        # Crear year block de df
        # Crear diccionari o algo de players i ranking de tants
        # Calcular unique dates in year_block
        # Per cada data
            # Crear 

In [57]:
k = 24
ksi = 400
s = 'delta'
initial_elo = 1500
min_games = 30
year_to_simulate = 'Tots'
rmv_retired = True

In [58]:
atp_matches_df = pd.read_csv('../web_data/clean_atp_matches.csv')
atp_players_df = pd.read_csv('../web_data/clean_atp_players.csv')
wta_matches_df = pd.read_csv('../web_data/clean_wta_matches.csv')
wta_players_df = pd.read_csv('../web_data/clean_wta_players.csv')
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{atp_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{atp_players_df.isna().sum()}'
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{wta_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{wta_players_df.isna().sum()}'

In [59]:
atp_ranking, atp_ranking_no_filtered, atp_elo_history, atp_elo_rankings_df  = simulate_tour(atp_matches_df, atp_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate, rmv_retired)
atp_ranking.index += 1
wta_ranking, wta_ranking_no_filtered, wta_elo_history, wta_elo_rankings_df = simulate_tour(wta_matches_df, wta_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate, rmv_retired)
wta_ranking.index += 1

0        1.0
1        2.0
2        NaN
3        NaN
4        3.0
        ... 
65984    NaN
65985    NaN
65986    NaN
65987    NaN
65988    NaN
Name: rank, Length: 65989, dtype: float64
0        NaN
1        NaN
2        NaN
3        NaN
4        1.0
        ... 
70031    NaN
70032    NaN
70033    NaN
70034    NaN
70035    NaN
Name: rank, Length: 70036, dtype: float64


In [60]:
atp_elo_history.to_csv('../web_data/atp_initial_elo_history.csv')
atp_ranking.to_csv('../web_data/atp_initial_ranking.csv')
atp_ranking_no_filtered.to_csv('../web_data/atp_initial_nofiltered_ranking.csv')
wta_elo_history.to_csv('../web_data/wta_initial_elo_history.csv')
wta_ranking.to_csv('../web_data/wta_initial_ranking.csv')
wta_ranking_no_filtered.to_csv('../web_data/wta_initial_nofiltered_ranking.csv')

In [61]:
atp_elo_rankings_df

,date,player_id,elo_rating,rank
0,1970-01-04,100100,1545.547546,1
1,1970-01-04,100087,1522.008400,2
2,1970-01-04,100037,1500.414778,3
3,1970-01-04,100084,1476.800015,4
4,1970-01-04,100058,1455.229261,5
...,...,...,...,...
194541,2024-12-18,209992,1608.204535,4
194542,2024-12-18,211663,1602.429103,5
194543,2024-12-18,210530,1552.083021,6
194544,2024-12-18,209414,1517.924702,7


In [62]:
len(atp_elo_history) / len(atp_matches_df)

2.0

In [63]:
atp_elo_history['date'].unique()

<DatetimeArray>
['1970-01-04 00:00:00', '1970-01-10 00:00:00', '1970-01-19 00:00:00',
 '1970-01-26 00:00:00', '1970-01-28 00:00:00', '1970-02-02 00:00:00',
 '1970-02-09 00:00:00', '1970-02-12 00:00:00', '1970-02-13 00:00:00',
 '1970-02-15 00:00:00',
 ...
 '2024-10-28 00:00:00', '2024-11-04 00:00:00', '2024-11-11 00:00:00',
 '2024-11-19 00:00:00', '2024-11-20 00:00:00', '2024-11-21 00:00:00',
 '2024-11-22 00:00:00', '2024-11-23 00:00:00', '2024-11-24 00:00:00',
 '2024-12-18 00:00:00']
Length: 3139, dtype: datetime64[ns]

In [64]:
atp_elo_rankings_df

,date,player_id,elo_rating,rank
0,1970-01-04,100100,1545.547546,1
1,1970-01-04,100087,1522.008400,2
2,1970-01-04,100037,1500.414778,3
3,1970-01-04,100084,1476.800015,4
4,1970-01-04,100058,1455.229261,5
...,...,...,...,...
194541,2024-12-18,209992,1608.204535,4
194542,2024-12-18,211663,1602.429103,5
194543,2024-12-18,210530,1552.083021,6
194544,2024-12-18,209414,1517.924702,7
